# 🧹 Phase 1: Data Cleaning & EDA
**Author:** Kyrylo Kudrevych

### 🎯 What are we doing here?
Welcome to the first step of the pipeline!

Before we can train any Machine Learning models, we have to prepare our dataset. Real-world web-scraped data is usually pretty messy, so our main goal in this notebook is to take raw text and turn it into a clean, 100% numeric table.

Here is exactly what we will do:
* **Extract:** Pull clean numbers out of weirdly formatted text strings.
* **Filter:** Remove extreme outliers and obvious human typos (like someone accidentally listing a 5 sq meter apartment).
* **Convert:** Turn text categories (like neighborhoods and floor levels) into numbers so our AI can actually understand them.

Let's dive into the data! 🚀

In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ast
import numpy as np

# Load the raw scraped data
df = pd.read_csv("../data/raw_rent_data.csv")
print(f"Initial rows: {df.count()[0]}")

## 🧹 1. Data Cleaning & Type Conversion

Real-world data is rarely perfect. The raw prices from our API are stuck inside text strings that look like dictionaries (e.g., `"{'value': 2500}"`).

To fix this, we will:
* Use Python to safely extract just the numerical values.
* Calculate our main target variable, `true_price`, which is the actual monthly cost of the apartment (**Total Rent + Administrative Rent**).
* Fill in missing values for basic amenities (like AC or balcony) with `False`.
* Group specific neighborhoods into broader categories (City Center, Mid-Tier, Outskirts) to help our model generalize better.

In [ ]:
def extract_value(price_string):
    if isinstance(price_string, (float, int)):
        return float(price_string)
    if pd.isna(price_string):
        return 0.0
    try:
        price_dict = ast.literal_eval(price_string)
        return float(price_dict.get('value', 0.0))
    except (ValueError, SyntaxError):
        return 0.0

df['total_price'] = df['total_price'].apply(extract_value)
df['rent_price'] = df['rent_price'].apply(extract_value)
df['true_price'] = df['total_price'] + df['rent_price']

df = df.dropna(subset=['total_price', 'location'])

amenity_cols = ['has_ac', 'has_balcony', 'has_terrace', 'has_parking', 'has_storage', 'is_secure']
df[amenity_cols] = df[amenity_cols].fillna(False)

df['floor'] = pd.to_numeric(df['floor'], errors='coerce')

df = df.fillna(0)

district_map = {
    'Jeżyce': 'City_Center',
    'Stare Miasto': 'City_Center',
    'Centrum': 'City_Center',
    'Wilda': 'Mid_Tier',
    'Grunwald': 'Mid_Tier',
    'Nowe Miasto': 'Mid_Tier',
    'Rataje': 'Mid_Tier',
    'Naramowice': 'Outskirts',
    'Piątkowo': 'Outskirts',
    'Winogrady': 'Mid_Tier',
    'Łacina': 'Mid_Tier',
    'Świerczewo': 'Outskirts',
    'Junikowo': 'Outskirts',
    'Kasztelanów': 'Outskirts',
    'Podolany': 'Outskirts'
}

# Apply the mapping
df['district_category'] = df['location'].map(district_map).fillna('Other')

## 🏢 2. Converting Text to Numbers (Ordinal Features)

Machine learning models only understand math, not words. Features like `floor` and `rooms` have a logical, natural order (e.g., the 5th floor is higher than the 1st floor).

We will map these text words (like `"FIRST"` or `"TWO"`) into sequential integers so the model can understand their mathematical relationship.

In [ ]:
floor_mapping = {
    'CELLAR': -1, 'GROUND': 0, 'FIRST': 1, 'SECOND': 2,
    'THIRD': 3, 'FOURTH': 4, 'FIFTH': 5, 'SIXTH': 6,
    'SEVENTH': 7, 'EIGHTH': 8, 'NINTH': 9, 'TENTH': 10,
    'ABOVE_TENTH': 11, 'GARRET': 12
}

rooms_mapping = {
    "ONE": 1, "TWO": 2, "THREE": 3, "FOUR": 4,
    "FIVE": 5, "SIX": 6, "SEVEN": 7
}

df['floor_num'] = df['floor'].map(floor_mapping)
df['rooms_num'] = df['rooms'].map(rooms_mapping)

## 📊 3. Exploratory Data Analysis (EDA) & Outlier Detection

Web-scraped data often contains human errors. For example, a user might accidentally type `10,000 PLN` for a normal apartment, or list a `5 sq meter` room.

To protect our model from learning from these extreme outliers, we will filter the dataset to standard, realistic boundaries:
* **Area:** Between 15 and 120 square meters.
* **Price:** Between 1,200 and 8,000 PLN.

Let's visualize the clean data to see what the Poznań rental market actually looks like!

In [ ]:
df = df[(df['area'] >= 15) & (df['area'] <= 120)]
df = df[(df['true_price'] >= 1200) & (df['true_price'] <= 8000)]

sns.histplot(df['true_price'], kde=True)
plt.title("Distribution of Rental Prices in Poznań")
plt.show()

sns.scatterplot(x="area", y="true_price", data=df)
plt.title("Area vs Price")
plt.show()

## 🗺️ 4. Preparing Categories (One-Hot Encoding)

Unlike floors or rooms, neighborhood names don't have a mathematical order (Jeżyce isn't mathematically "greater than" Wilda).

To feed these locations into our model, we use a technique called **One-Hot Encoding** (`pd.get_dummies`).
* This creates a new binary column (1 or 0) for every district.
* Finally, we drop the old text columns and any columns we don't need for training to keep our table clean.

In [ ]:
df = pd.get_dummies(df, columns=['location'], dtype=int)
df = pd.get_dummies(df, columns=['district_category'], dtype=int)
df = df.drop(columns=['id', 'title', 'total_price', 'rent_price', 'floor', 'rooms'])

## 💾 5. Saving the Clean Data

Our data is now completely numeric, cleaned of outliers, and ready for machine learning!

We will save it as a `.parquet` file. This format is much faster and smaller than CSV, and it perfectly preserves our column data types so we can easily load it into our next notebook.

In [ ]:
df.to_parquet('notebooks_data/data_ready_for_ml.parquet')

## 📈 6. Model Evaluation & Feature Importance
Let's see how well our model performed by transforming our predictions back to standard PLN and comparing them to reality.

*(Note: As seen in the scatterplot, the model experiences slight heteroscedasticity-meaning it becomes slightly less accurate on ultra-luxury, high-priced apartments, which is a common behavior in real estate modeling).*

In [ ]:
# plt.figure(figsize=(10, 6))
# sns.scatterplot(x=final_y_test, y=final_ensemble_predictions, alpha=0.7)
# plt.plot([1200, 8000], [1200, 8000], color='red', linestyle='--')
# plt.xlabel('Actual Price (PLN)')
# plt.ylabel('Predicted Price (PLN)')
# plt.title('AI Predictions vs Reality in Poznań')
# plt.show()
#
# importances = model.feature_importances_
# feature_names = X_train.columns
# feature_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
# feature_df = feature_df.sort_values(by='Importance', ascending=False)
# print("\n--- Top 10 Drivers of Real Estate Prices ---")
# print(feature_df.head(10))

## 💡 7. Conclusion & Key Findings

By refactoring our data processing and training steps, we established a robust, leak-free Data Science pipeline.

**Key Takeaways:**
1. **Solid Accuracy:** Our optimized Random Forest achieved a Mean Absolute Error (MAE) of **~337 PLN**. This means that, on average, our model can predict the true monthly cost of a Poznań apartment within about 340 PLN of its actual listed price.
2. **Space is King:** Our Feature Importance analysis reveals that the physical `area` of the apartment is by far the most dominant driver of rent prices, accounting for over 55% of the model's decision-making weight.
3. **Location Premiums:** Geographic orientation heavily dictates pricing tiers. The engineered location flags show that living in central, popular districts like *Łazarz*, *Jeżyce*, and *Stare Miasto* carries a distinct premium compared to the outskirts.

The model is now ready to be deployed as the backend engine for the **Poznań Rent Radar UI**!